# 05 — Market Basket Analysis (Apriori)

**Goal:** Discover which skills frequently co-occur in job postings using the Apriori algorithm (Market Basket Analysis).

## Pipeline Overview
1. Load raw skill data → build wide-format (one row per job, list of skills)
2. One-hot encode → boolean skill matrix
3. Run Apriori to find frequent itemsets
4. Extract association rules (lift ≥ 1.2, confidence ≥ 0.5, with auto-fallback)
5. Visualize top rules (bar chart + heatmap)
6. Save `data/clean/mba_rules.csv` for Streamlit Tab 5

**Input:** `data/raw/skills_demand.csv`, `data/clean/primary_wide.csv`  
**Output:** `data/clean/mba_rules.csv`, `assets/charts/06_mba_top_rules.png`, `assets/charts/07_mba_rules_heatmap.png`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from market_basket import run_mba_pipeline, load_mba_rules

RAW_PATH    = PROJECT_ROOT / 'data' / 'raw'   / 'skills_demand.csv'
WIDE_PATH   = PROJECT_ROOT / 'data' / 'clean' / 'primary_wide.csv'
ONEHOT_PATH = PROJECT_ROOT / 'data' / 'clean' / 'primary_onehot.csv'
RULES_PATH  = PROJECT_ROOT / 'data' / 'clean' / 'mba_rules.csv'
CHARTS_DIR  = PROJECT_ROOT / 'assets' / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print('Paths configured ✅')

## Phase 1 — Run Apriori Pipeline

In [ ]:
itemsets, rules = run_mba_pipeline(
    raw_path=RAW_PATH,
    wide_path=WIDE_PATH,
    onehot_path=ONEHOT_PATH,
    top_n=30
)

print(f"Frequent itemsets: {len(itemsets)}")
print(f"Association rules: {len(rules)}")
itemsets.sort_values('support', ascending=False).head(20)

## Phase 2 — Inspect & Confirm Rules

In [ ]:
assert len(rules) >= 5, f"Expected ≥5 rules, got {len(rules)}"
print(f"✅ {len(rules)} rules survived the filter")
rules[['antecedents','consequents','support','confidence','lift']].head(10)

In [ ]:
# Spot-check top rule
top = rules.iloc[0]
print(f"Top rule : {top['antecedents']} → {top['consequents']}")
print(f"  Support    : {top['support']:.4f}")
print(f"  Confidence : {top['confidence']:.4f}")
print(f"  Lift       : {top['lift']:.4f}")

## Phase 3 — Save Rules

In [ ]:
rules.to_csv(RULES_PATH, index=False)
print(f"✅ Saved {len(rules)} rules to {RULES_PATH}")

## Phase 4 — Visualization: Bar Chart

In [ ]:
top10 = rules.head(10).copy()
top10['rule_label'] = top10['antecedents'] + " → " + top10['consequents']

plt.figure(figsize=(10, 6))
sns.barplot(data=top10, y='rule_label', x='lift', hue='rule_label', legend=False, palette='Blues_r')
plt.title(
    f"Strongest Skill Associations — Top Rule: "
    f"{top10.iloc[0]['rule_label']} (Lift {top10.iloc[0]['lift']:.2f})"
)
plt.xlabel('Lift Score')
plt.tight_layout()
plt.savefig(CHARTS_DIR / '06_mba_top_rules.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Bar chart saved')

## Phase 5 — Visualization: Heatmap

In [ ]:
pivot = rules.pivot_table(index='antecedents', columns='consequents', values='lift')

plt.figure(figsize=(10, 8))
sns.heatmap(pivot, annot=True, cmap='Blues', fmt='.2f', cbar_kws={'label': 'Lift Score'})
plt.title('Skill Association Lift Heatmap')
plt.xlabel('Consequents')
plt.ylabel('Antecedents')
plt.tight_layout()
plt.savefig(CHARTS_DIR / '07_mba_rules_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap saved')

## Summary

| Output | Location |
|:-------|:---------|
| Frequent itemsets | in-memory `itemsets` DataFrame |
| Association rules | `data/clean/mba_rules.csv` |
| Bar chart | `assets/charts/06_mba_top_rules.png` |
| Lift heatmap | `assets/charts/07_mba_rules_heatmap.png` |

These outputs feed directly into **Streamlit Tab 5 — 🛒 Skill Associations**.